## Circle Packing

Pack circles of given radii to minimize overlap and overall bounding box size, without any nesting or graph structure.

In [ ]:
%config InlineBackend.figure_formats = ['svg']

In [ ]:
import numpy as np
from matplotlib import patches as mpatches
from matplotlib import pyplot as plt
from vizopt.templates import circle_packing

In [ ]:
rng = np.random.default_rng(0)
n_circles = 30
radii = rng.uniform(0.01, 1, size=30).tolist()
initial_node_xys = np.random.rand(n_circles, 2).astype(np.float32) * 5

In [ ]:
def plot_circles(positions, radii):
    fig, ax = plt.subplots(figsize=(5, 5))
    colors = plt.cm.tab20.colors
    for i, (xy, r) in enumerate(zip(positions, radii)):
        ax.add_patch(mpatches.Circle(xy, radius=r, color=colors[i % len(colors)], alpha=0.6, ec="black", linewidth=0.5))

    margin = max(radii)
    ax.set_xlim(min(p[0] for p in positions) - margin, max(p[0] for p in positions) + margin)
    ax.set_ylim(min(p[1] for p in positions) - margin, max(p[1] for p in positions) + margin)
    ax.set_aspect("equal")
    ax.set_title(f"Circle packing: {len(radii)} circles")
    plt.axis("off")
    plt.tight_layout()

plot_circles(initial_node_xys, radii)

In [ ]:
positions = circle_packing.optimize_circle_packing(
    radii=radii,
    weight_total_size=10.,
    collision_offset=0.1,
    optim_kwargs={"n_iters": 5000, "learning_rate": 0.01},
)

The optimizer compresses x and y independently (the total size penalty treats each axis separately), so circles are pushed into a compact rectangular arrangement rather than a circular cluster.

In [ ]:
plot_circles(positions, radii)

## Animation

In [ ]:
from IPython.display import HTML
from vizopt.animation import SnapshotCallback, animate
from vizopt.templates.circle_packing import build_circle_packing_problem

problem = build_circle_packing_problem(
    radii=radii,
    weight_total_size=10.0,
    collision_offset=0.1,
    initial_node_xys=initial_node_xys,
)

cb = SnapshotCallback(every=100)
problem.optimize(n_iters=5000, learning_rate=0.01, callback=cb)

In [ ]:
anim = animate(problem, cb.snapshots, interval=100)
HTML(anim.to_jshtml())

In [ ]:
import numpy as np
from pathlib import Path
from IPython.display import SVG


In [ ]:
def snapshots_to_animated_svg(snapshots, input_params, fps=10, size=500, calc_mode="linear"):
    radii = input_params["node_radii"]
    n_circles = len(radii)
    n_frames = len(snapshots)
    tab20 = [
        "#1f77b4","#aec7e8","#ff7f0e","#ffbb78","#2ca02c",
        "#98df8a","#d62728","#ff9896","#9467bd","#c5b0d5",
        "#8c564b","#c49c94","#e377c2","#f7b6d2","#7f7f7f",
        "#c7c7c7","#bcbd22","#dbdb8d","#17becf","#9edae5",
    ]

    all_xy = np.concatenate([v["node_xys"] for _, v in snapshots], axis=0)
    margin = float(max(radii))
    xmin = float(all_xy[:, 0].min()) - margin
    ymin = float(all_xy[:, 1].min()) - margin
    span = max(
        float(all_xy[:, 0].max()) + margin - xmin,
        float(all_xy[:, 1].max()) + margin - ymin,
    )

    def to_svg(x, y):
        return (x - xmin) / span * size, (span - (y - ymin)) / span * size

    total_dur = n_frames / fps

    # linear requires keyTimes to end at 1.0; append first frame to close the loop
    if calc_mode == "linear":
        key_times = ";".join(f"{fi / n_frames:.6f}" for fi in range(n_frames)) + ";1.000000"
        extra = 1  # one extra value (wrap back to frame 0)
    else:
        key_times = ";".join(f"{fi / n_frames:.6f}" for fi in range(n_frames))
        extra = 0

    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{size}" height="{size}">',
        f'  <rect width="{size}" height="{size}" fill="white"/>',
    ]
    for i in range(n_circles):
        cx_all = [
            f"{to_svg(float(v['node_xys'][i, 0]), float(v['node_xys'][i, 1]))[0]:.1f}"
            for _, v in snapshots
        ]
        cy_all = [
            f"{to_svg(float(v['node_xys'][i, 0]), float(v['node_xys'][i, 1]))[1]:.1f}"
            for _, v in snapshots
        ]
        cx_vals = ";".join(cx_all + cx_all[:1] * extra)
        cy_vals = ";".join(cy_all + cy_all[:1] * extra)
        sr = float(radii[i]) / span * size
        color = tab20[i % len(tab20)]
        lines.append(
            f'  <circle r="{sr:.1f}" fill="{color}" fill-opacity="0.6" stroke="black" stroke-width="0.5">'
        )
        lines.append(
            f'    <animate attributeName="cx" calcMode="{calc_mode}"'
            f' values="{cx_vals}" keyTimes="{key_times}"'
            f' dur="{total_dur:.3f}s" repeatCount="indefinite"/>'
        )
        lines.append(
            f'    <animate attributeName="cy" calcMode="{calc_mode}"'
            f' values="{cy_vals}" keyTimes="{key_times}"'
            f' dur="{total_dur:.3f}s" repeatCount="indefinite"/>'
        )
        lines.append('  </circle>')
    lines.append('</svg>')
    return '\n'.join(lines)

svg_str = snapshots_to_animated_svg(cb.snapshots, problem.input_parameters, fps=10)
Path("circle_packing_animated.svg").write_text(svg_str)
SVG(data=svg_str)